# EDA d'inspection — leboncoin-private

Dataset candidat pour le pilier Prix. Cible = `prix_eur`.
Annonces **de particuliers** (`owner_type=private`), collectées par `collecte/scraper/scraper.py`
puis rapatriées par `collecte/scraper/dataset.py`.

> ⚠️ **Gate EDA** (cf. `ml/AGENTS.md`) : ce notebook **inspecte** seulement.
> Aucun nettoyage, aucune modélisation. Le fichier `raw/` n'est jamais modifié.
> À la fin : résumé chiffré → **STOP** → validation avant toute suite.

> Les **photos** du même dataset sont inspectées à part, côté pilier État :
> `dl/notebooks/leboncoin-private/01_eda_inspection.ipynb`. Deux piliers, deux Gates.

In [1]:
import json
from pathlib import Path

import pandas as pd

RAW = Path("../../data/leboncoin-private/raw/annonces.parquet")
df = pd.read_parquet(RAW)
print(f"{df.shape[0]} lignes x {df.shape[1]} colonnes")

20915 lignes x 24 colonnes


## 1. Colonnes & types

In [2]:
print("Colonnes:")
for col in df.columns:
    print(f"  - {col}")
print()
df.dtypes

Colonnes:
  - pk
  - source
  - ad_id
  - url
  - etat
  - etat_label
  - owner_type
  - prix_eur
  - marque
  - modele
  - annee
  - kilometrage
  - energie
  - boite
  - ville
  - code_postal
  - departement
  - region
  - nb_images
  - images
  - image_keys
  - raw
  - scraped_at
  - inserted_at



pk                               int64
source                             str
ad_id                              str
url                                str
etat                               str
etat_label                         str
owner_type                         str
prix_eur                       float64
marque                             str
modele                             str
annee                            Int64
kilometrage                      Int64
energie                            str
boite                              str
ville                              str
code_postal                        str
departement                        str
region                             str
nb_images                        Int64
images                          object
image_keys                      object
raw                                str
scraped_at     datetime64[us, Etc/UTC]
inserted_at    datetime64[us, Etc/UTC]
dtype: object

In [3]:
df.head(3)

,pk,source,ad_id,url,etat,etat_label,owner_type,prix_eur,marque,modele,...,ville,code_postal,departement,region,nb_images,images,image_keys,raw,scraped_at,inserted_at
0,1,leboncoin,3223301080,https://www.leboncoin.fr/ad/voitures/3223301080,undamaged,Non endommagé,private,14600.0,MERCEDES-BENZ,Classe R,...,Marseille,13013,Bouches-du-Rhône,Provence-Alpes-Côte d'Azur,9,[https://img.leboncoin.fr/api/v1/lbcpb1/images...,[scraping/leboncoin/undamaged/3223301080/00.jp...,"{""id"": 3223301080, ""url"": ""https://www.lebonco...",2026-07-27 13:34:09.699000+00:00,2026-07-27 13:34:09.936340+00:00
1,2,leboncoin,3234385109,https://www.leboncoin.fr/ad/voitures/3234385109,undamaged,Non endommagé,private,6000.0,MERCEDES-BENZ,Classe SLK,...,Strasbourg,67000,Bas-Rhin,Alsace,3,[https://img.leboncoin.fr/api/v1/lbcpb1/images...,[scraping/leboncoin/undamaged/3234385109/00.jp...,"{""id"": 3234385109, ""url"": ""https://www.lebonco...",2026-07-27 13:34:09.699000+00:00,2026-07-27 13:34:09.936340+00:00
2,3,leboncoin,3240311828,https://www.leboncoin.fr/ad/voitures/3240311828,undamaged,Non endommagé,private,10990.0,RENAULT,Captur,...,Meyzieu,69330,Rhône,Rhône-Alpes,1,[https://img.leboncoin.fr/api/v1/lbcpb1/images...,[scraping/leboncoin/undamaged/3240311828/00.jpg],"{""id"": 3240311828, ""url"": ""https://www.lebonco...",2026-07-27 13:34:09.699000+00:00,2026-07-27 13:34:09.936340+00:00


## 2. Valeurs manquantes (% par colonne)

In [4]:
na_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(na_pct[na_pct > 0].to_string() if (na_pct > 0).any() else "aucune colonne avec des manquants")

image_keys     67.7
departement     2.3


## 3. Qualité de la cible `prix_eur`

On vérifie le type, les valeurs non plausibles et la distribution — sans rien filtrer.
Deux seuils sont regardés au passage :

- le bas de la distribution (annonces à quelques euros = prix d'appel, pièces détachées, erreurs) ;
- **50 000 €**, périmètre produit fixé par l'**ADR 0002**.

In [5]:
prix = df["prix_eur"]
print("dtype        :", prix.dtype)
print("Manquants    :", int(prix.isna().sum()))
print("Non positifs :", int((prix <= 0).sum()))
print()
print("Describe :")
print(prix.describe().round(0).to_string())
print()
print("Quantiles :")
print(prix.quantile([0.01, 0.05, 0.50, 0.90, 0.95, 0.99]).round(0).to_string())
print()
for seuil in (1, 100, 500):
    print(f"prix <= {seuil:>5} EUR : {int((prix <= seuil).sum()):>5}")
print()
hors_scope = int((prix > 50_000).sum())
print(f"prix > 50 000 EUR (hors perimetre ADR 0002) : {hors_scope} ({100 * hors_scope / len(df):.1f} %)")
print("10 prix les plus hauts :", sorted(prix.dropna())[-10:])

dtype        : float64
Manquants    : 0
Non positifs : 0

Describe :
count     20915.0
mean       7799.0
std       12616.0
min           1.0
25%        1800.0
50%        4350.0
75%        9990.0
max      999999.0

Quantiles :
0.01      400.0
0.05      650.0
0.50     4350.0
0.90    18490.0
0.95    24993.0
0.99    45000.0

prix <=     1 EUR :    24
prix <=   100 EUR :    79
prix <=   500 EUR :   702

prix > 50 000 EUR (hors perimetre ADR 0002) : 147 (0.7 %)
10 prix les plus hauts : [156000.0, 160000.0, 164800.0, 185000.0, 200000.0, 222000.0, 230000.0, 239990.0, 250000.0, 999999.0]


## 4. Aberrations & doublons

In [6]:
print("ad_id unique          :", df["ad_id"].is_unique)
print("Lignes strictement dupliquees :", int(df.duplicated(subset=[c for c in df.columns
                                                                  if c not in ("pk", "raw", "images",
                                                                               "image_keys")]).sum()))
quintuplet = ["marque", "modele", "annee", "kilometrage", "prix_eur"]
print(f"Quasi-doublons {quintuplet} :", int(df.duplicated(subset=quintuplet).sum()))
print()
km = df["kilometrage"]
print("km min / median / max :", int(km.min()), "/", int(km.median()), "/", int(km.max()))
print("km >= 999 999 (sentinelle probable) :", int((km >= 999_999).sum()))
print("km <= 100                           :", int((km <= 100).sum()))
print()
an = df["annee"]
print("annee min / max        :", int(an.min()), "/", int(an.max()))
print("annee < 1980           :", int((an < 1980).sum()))
print("annee > 2026           :", int((an > 2026).sum()))

ad_id unique          : True
Lignes strictement dupliquees : 0
Quasi-doublons ['marque', 'modele', 'annee', 'kilometrage', 'prix_eur'] : 71

km min / median / max : 1 / 180000 / 999999
km >= 999 999 (sentinelle probable) : 4
km <= 100                           : 43

annee min / max        : 1960 / 2026
annee < 1980           : 489
annee > 2026           : 0


## 5. Cardinalité des catégorielles clés

Point de vigilance repris de l'**ADR 0001** : les **électriques purs** avaient été retirés du
périmètre sur le dataset précédent (cote atypique, volume trop faible pour être modélisée).
La question se repose ici.

In [7]:
for col in ["marque", "modele", "energie", "boite", "region"]:
    print(f"--- {col} : {df[col].nunique()} valeurs distinctes ---")
    print(df[col].value_counts(dropna=False).head(6).to_string())
    print()

--- marque : 97 valeurs distinctes ---
marque
PEUGEOT       3291
RENAULT       3278
CITROEN       2162
VOLKSWAGEN    1860
BMW           1353
AUDI          1151

--- modele : 926 valeurs distinctes ---
modele
Clio       1088
Golf        732
Megane      600
206         464
308         452
Série 3     452

--- energie : 8 valeurs distinctes ---
energie
Diesel                  11644
Essence                  8418
Hybride                   405
Électrique                253
Hybride Rechargeable       82
GPL                        82

--- boite : 2 valeurs distinctes ---
boite
Manuelle       15213
Automatique     5702

--- region : 26 valeurs distinctes ---
region
Ile-de-France                 2648
Rhône-Alpes                   2473
Provence-Alpes-Côte d'Azur    1826
Nord-Pas-de-Calais            1276
Pays de la Loire              1155
Aquitaine                     1127



## 6. État véhicule déclaré (`etat`) — la colonne nouvelle

Cette colonne n'existe sur aucun des candidats précédents. Deux questions :

1. **Est-elle un signal de décote exploitable** pour le modèle Prix ?
2. Sa répartition reflète-t-elle le marché ? **Non** : la collecte est un échantillonnage
   *stratifié délibéré* (`ETATS`, `collecte/scraper/scraper.py`) — les classes rares sont prises en
   entier, les fréquentes plafonnées. La distribution ci-dessous est donc **construite**, pas subie.

In [8]:
print("Repartition :")
print(df["etat"].value_counts(dropna=False).to_string())
print()

REF_YEAR = 2026  # collecte 06-07/2026
tmp = df.assign(age=REF_YEAR - df["annee"])
resume = tmp.groupby("etat").agg(
    n=("prix_eur", "size"),
    prix_median=("prix_eur", "median"),
    km_median=("kilometrage", "median"),
    age_median=("age", "median"),
).sort_values("prix_median", ascending=False)
print("Croisement etat x prix / km / age (medianes) :")
print(resume.round(0).to_string())

Repartition :
etat
major_repairs_needed      3436
good_overall_condition    3383
minor_repairs_needed      3366
normal_wear_and_tear      3362
undamaged                 3359
excellent_condition       2046
not_drivable              1222
damaged                    741

Croisement etat x prix / km / age (medianes) :
                           n  prix_median  km_median  age_median
etat                                                            
excellent_condition     2046      16900.0    89150.0         7.0
undamaged               3359       9000.0   146500.0        12.0
good_overall_condition  3383       7000.0   160000.0        14.0
normal_wear_and_tear    3362       4500.0   185000.0        16.0
minor_repairs_needed    3366       2500.0   213000.0        19.0
major_repairs_needed    3436       1500.0   228000.0        20.0
damaged                  741       1400.0   214000.0        19.0
not_drivable            1222       1400.0   211000.0        21.0


## 7. Inventaire du champ `raw`

`raw` conserve le JSON complet de l'annonce (colonne texte). Il contient un bloc `attributes` bien
plus riche que les colonnes chaudes : de quoi construire des features supplémentaires. On mesure ici
le **taux de remplissage** de chaque attribut, seul critère qui décide de son utilité.

Deux champs méritent un regard à part : `body` (la description) et `first_publication_date`.

In [9]:
from collections import Counter

raws = [json.loads(s) for s in df["raw"]]

compteur = Counter()
for r in raws:
    compteur.update(r.get("attributes", {}).keys())

n = len(df)
print(f"{len(compteur)} attributs distincts sur {n} annonces")
print()
print(f"{'attribut':<38}{'n':>7}{'%':>7}")
for cle, val in compteur.most_common():
    if val / n >= 0.05:  # sous 5 % de remplissage, inexploitable en feature
        print(f"{cle:<38}{val:>7}{100 * val / n:>6.0f}%")

40 attributs distincts sur 20915 annonces

attribut                                    n      %
fuel                                    20915   100%
brand                                   20915   100%
model                                   20915   100%
gearbox                                 20915   100%
mileage                                 20915   100%
regdate                                 20915   100%
is_import                               20915   100%
u_car_brand                             20915   100%
u_car_model                             20915   100%
vehicle_damage                          20915   100%
vehicle_vsp                             20912   100%
vehicule_color                          20799    99%
doors                                   20796    99%
seats                                   20785    99%
vehicle_is_eligible_p2p                 20759    99%
licence_plate_available                 20741    99%
vehicle_type                            20720    99%
iss

In [10]:
bodies = pd.Series([(r.get("body") or "").strip() for r in raws])
vides = int((bodies == "").sum())
print(f"body vide : {vides} / {n}  ({100 * vides / n:.1f} %)")
print("-> la description n'est pas rendue sur le listing (limite connue, cf. collecte/scraper/README.md).")
print("   Aucune feature texte possible sans re-scraper chaque page annonce.")
print()

fpd = pd.to_datetime(pd.Series([r.get("first_publication_date") for r in raws]), errors="coerce")
print(f"first_publication_date renseigne : {int(fpd.notna().sum())} / {n}")
print("  du", fpd.min(), "au", fpd.max())
print("-> matiere premiere du pilier Date (delai de vente). Hors perimetre immediat, a ne pas perdre.")

body vide : 20915 / 20915  (100.0 %)
-> la description n'est pas rendue sur le listing (limite connue, cf. collecte/scraper/README.md).
   Aucune feature texte possible sans re-scraper chaque page annonce.

first_publication_date renseigne : 20915 / 20915
  du 2026-05-28 19:59:25 au 2026-07-27 20:51:44
-> matiere premiere du pilier Date (delai de vente). Hors perimetre immediat, a ne pas perdre.


## 8. Résumé chiffré (→ STOP validation)

In [11]:
na_global = df.isna().mean().mean() * 100
tres_vides = int(((df.isna().mean() * 100) > 90).sum())
elec = int((df["energie"] == "Électrique").sum())

print("=" * 62)
print("RESUME — leboncoin-private")
print("=" * 62)
print(f"Dimensions            : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"Vendeurs              : {df['owner_type'].unique().tolist()} (100 % particuliers)")
print(f"NA global moyen       : {na_global:.1f} %   | colonnes >90% vides : {tres_vides}")
print(f"Cible prix_eur        : {prix.dtype}, {int((prix <= 0).sum())} non positifs, "
      f"{int(prix.isna().sum())} manquants")
print(f"prix median / max     : {prix.median():.0f} / {prix.max():.0f} EUR")
print(f"prix > 50k (ADR 0002) : {int((prix > 50_000).sum())}   | prix <= 100 EUR : {int((prix <= 100).sum())}")
print(f"km sentinelle 999 999 : {int((df['kilometrage'] >= 999_999).sum())}   | annee < 1980 : {int((df['annee'] < 1980).sum())}")
print(f"ad_id unique          : {df['ad_id'].is_unique}   | quasi-doublons vehicule : "
      f"{int(df.duplicated(subset=quintuplet).sum())}")
print(f"Cardinalite           : {df['marque'].nunique()} marques, {df['modele'].nunique()} modeles, "
      f"{df['region'].nunique()} regions")
print(f"Electriques purs      : {elec} (cf. ADR 0001)")
print(f"Etat declare          : {df['etat'].nunique()} classes, 0 manquant — echantillonnage stratifie")
print(f"Description (body)    : vide sur {vides} / {n} — aucune feature texte")
print(f"Annonces avec photos  : {int(df['image_keys'].notna().sum())} / {n} (quota volontaire)")
print("=" * 62)
print("STOP — attente validation avant nettoyage / modelisation.")

RESUME — leboncoin-private
Dimensions            : 20915 lignes x 24 colonnes
Vendeurs              : ['private'] (100 % particuliers)
NA global moyen       : 2.9 %   | colonnes >90% vides : 0
Cible prix_eur        : float64, 0 non positifs, 0 manquants
prix median / max     : 4350 / 999999 EUR
prix > 50k (ADR 0002) : 147   | prix <= 100 EUR : 79
km sentinelle 999 999 : 4   | annee < 1980 : 489
ad_id unique          : True   | quasi-doublons vehicule : 71
Cardinalite           : 97 marques, 926 modeles, 26 regions
Electriques purs      : 253 (cf. ADR 0001)
Etat declare          : 8 classes, 0 manquant — echantillonnage stratifie
Description (body)    : vide sur 20915 / 20915 — aucune feature texte
Annonces avec photos  : 6762 / 20915 (quota volontaire)
STOP — attente validation avant nettoyage / modelisation.
